In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
import torch.nn.functional as F
from torch.nn import *
import torch
from tqdm import tqdm
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
import itertools
from constants import *
from model_gnn import GNN
import gc

# DEVICE1/DEVICE2 imported from constants

In [ ]:
donor_df = pd.read_parquet('data/pci_db_donors.parquet')

# Cross-val / test splits
kf = KFold(n_splits=6, shuffle=True, random_state=42)
splits = [[donor_df.index[train_ids].values, donor_df.index[test_ids].values] for train_ids, test_ids in kf.split(donor_df.index)]

donor_df

In [ ]:
mhc_df = pd.read_parquet('data/mhc_db.parquet')
mhc_df = mhc_df.loc[mhc_df['allele'].isin(donor_df.columns)]
mhc_df['sequence'] = mhc_df['sequence'].apply(lambda x: ' '.join(list(x)))
mhc_homo = pd.DataFrame([['HLA-A*homozygous', 'HLA-A*homozygous'], ['HLA-B*homozygous', 'HLA-B*homozygous'], ['HLA-C*homozygous', 'HLA-C*homozygous']], columns=mhc_df.columns)
mhc_df = pd.concat([mhc_df, mhc_homo])
mhc_df['sequence'] = '[CLS] ' + mhc_df['sequence'] + ' [SEP]'
mhc_df

In [ ]:
tue_db = pd.read_parquet('data/pci_db.parquet')

tue_db = tue_db.loc[tue_db['donor_code'].isin(donor_df.index)]
tue_db = tue_db.loc[tue_db['mhc_class'] == 'I']
tue_db['peptide'] = ['[CLS] ' + ' '.join(list(peptide)) + ' [SEP]' for peptide in tue_db['peptide'].values]
tue_db

In [ ]:
donor_df = donor_df.loc[donor_df.index.isin(tue_db['donor_code'].unique())]
donor_df

In [ ]:
MAX_LEN_MHC = mhc_df['sequence'].str.split().str.len().max()
MAX_LEN_PEP = pd.Series(tue_db['peptide'].unique()).str.split().str.len().max()
MAX_LEN_PEP, MAX_LEN_MHC

In [ ]:
mhc_features = np.array([
    [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_MHC - len(seq.split()))] for seq in mhc_df['sequence'].values
])

In [ ]:
def get_hetero_data(donor_list):
    data = []
    mhc_x = torch.tensor(mhc_features, dtype=torch.int32)
    mhc_mhc_edges = torch.concat([torch.arange(len(mhc_features)).unsqueeze(0), torch.arange(len(mhc_features)).unsqueeze(0)], 0)  # MHCs only connected to themselves
    for donor in tqdm(donor_list):
        slice_df = tue_db.loc[tue_db['donor_code'] == donor]
        mhc_y = torch.tensor(donor_df.loc[donor].values, dtype=torch.float32).unsqueeze(1)
        for sample in sorted(slice_df['sample_code'].unique()):
            sample_slice_df = slice_df.loc[slice_df['sample_code'] == sample]
            sample_peptide_features = np.array([
                [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_PEP - len(seq.split()))] for seq in sample_slice_df['peptide'].values
            ])
            
            entry = HeteroData()
            entry['peptide'].x = torch.tensor(sample_peptide_features, dtype=torch.int32)
            entry['mhc'].x = mhc_x
            entry['mhc'].y = mhc_y
            
            peptide_mhc_edges = torch.tensor(np.array(list(itertools.product(range(len(entry['peptide'].x)), range(len(mhc_features))))).T, dtype=torch.int32)
            peptide_peptide_edges = torch.concat([torch.arange(len(entry['peptide'].x)).unsqueeze(0), torch.arange(len(entry['peptide'].x)).unsqueeze(0)], 0)
            entry['peptide', 'determines', 'mhc'].edge_index = peptide_mhc_edges
            entry['peptide', 'influences', 'peptide'].edge_index = peptide_peptide_edges
            entry['mhc', 'influences', 'mhc'].edge_index = mhc_mhc_edges
            entry.donor_code = donor
            entry.sample_code = sample
            data.append(entry)
    return data

## Inner CV Validation

In [ ]:
import copy

gc.collect()
torch.cuda.empty_cache()

# works only with batch size == 1
def get_typing(batch, pred):
    tmp = pd.DataFrame(np.concatenate([
        pred.cpu().detach().numpy(),
        batch['mhc'].y.cpu().detach().numpy(),
        batch['mhc'].batch.unsqueeze(1)
    ], axis=1), columns=['y_pred', 'y', 'batch'])
    tmp['allele'] = np.concatenate(np.repeat([donor_df.columns.values], batch.batch_size, axis=0))
    tmp['locus'] = tmp['allele'].str[4]
    
    tmp = tmp.groupby(['batch', 'locus']).apply(
        lambda x: pd.Series({
            'y': ','.join(x.sort_values('y')['allele'][-2:].values),
            'y_pred': ','.join(x.sort_values('y_pred')['allele'][-2:].values),
        }), include_groups=False
    ).reset_index()
    tmp['acc'] = tmp.apply(lambda x: len(set(x['y'].split(',')).intersection(set(x['y_pred'].split(',')))) / 2, axis=1)
    tmp['donor_code'] = batch.donor_code[0]
    tmp['sample_code'] = batch.sample_code[0]
    return tmp
    

NAME = 'train_mhc_ba_el_in-silico_pci_db'
EPOCHS = 30
BATCH_SIZE = 4
MAX_PEPTIDES_TRAIN = 1_000

val_df = []
for fold, (train_val_donors, _) in enumerate(splits):
    print(f'Fold {fold}')
    
    train_donors, val_donors = train_test_split(train_val_donors, train_size=0.8, shuffle=True, random_state=42)
    
    tue_train_data = get_hetero_data(train_donors)
    tue_val_data = get_hetero_data(val_donors)
    
    model = GNN(
        vocab_size=45,
        embedding_dim=128,
        dim_ff_enc_pep=128,
        n_heads_enc_pep=8,
        n_layers_enc_pep=6,
        dim_ff_enc_mhc=1024,
        n_heads_enc_mhc=8,
        n_layers_enc_mhc=6,
        n_heads_conv=8,
        n_layers_conv=2,
        dim_out_conv=32,
        dropout=0.1,
        act=F.leaky_relu,
        device1=DEVICE1,
        device2=DEVICE2,
    )
    
    state_dict = torch.load('weights/in_silico/pretrain_mhc_ba_el_in-silico.pt')
    model.load_state_dict(state_dict, strict=True)
        
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    val_loader = DataLoader(tue_val_data, batch_size=1, shuffle=False)
    
    # Training loop
    for epoch in range(1, EPOCHS+1):
        train_batched_data = [
            b.subgraph({
                'peptide': np.random.choice(
                    len(b['peptide'].x), min(int(len(b['peptide'].x)), MAX_PEPTIDES_TRAIN), replace=False
                )
            }) for b in tue_train_data
        ]
        train_loader = DataLoader(train_batched_data, batch_size=BATCH_SIZE, shuffle=True)
        model.train()
        batches = tqdm(train_loader)
        bl = []
        for batch in batches:
            optimizer.zero_grad()
            out = model(batch)
            loss = F.binary_cross_entropy_with_logits(out, batch['mhc'].y.to(DEVICE2))
            bl.append(loss.item())
            loss.backward()
            optimizer.step()
            descr = f'Train epoch {epoch} - loss: {np.mean(bl[-100:]):.4f}'
            batches.set_description(descr)
    
        if epoch % 10 == 0:
            eval_model = copy.deepcopy(model).to('cpu')
            eval_model.device1 = 'cpu'
            eval_model.device2 = 'cpu'
            eval_model.eval()
            with torch.no_grad():
                batches = tqdm(val_loader)
                for batch in batches:
                    out = eval_model(batch)
                    loss = F.binary_cross_entropy_with_logits(out, batch['mhc'].y.to('cpu'))
                    batch_df = get_typing(batch, out)
                    batch_df['loss'] = loss.cpu().detach().numpy()
                    batch_df['fold'] = fold
                    batch_df['epoch'] = epoch
                    val_df.append(batch_df)
    
val_df = pd.concat(val_df)
val_df['homozygous'] = val_df['y'].str.contains('homozygous')
val_df.to_csv(f'validation/cross_val_pci_db/{NAME}.csv', index=False)

# Outer CV Testing

### Training

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# works only with batch size >= 2
def get_accuracy(batch, pred):
    acc = []
    ptr = batch['mhc']['ptr'][1:]
    for i in range(1, len(ptr)):
        p = pred[ptr[i-1]:ptr[i]].cpu().detach().numpy().squeeze()
        y = batch['mhc'].y[ptr[i-1]:ptr[i]].cpu().detach().numpy().squeeze()
        acc.append(len(set(np.argsort(y)[-6:]).intersection(set(np.argsort(p)[-6:]))) / 6)
    return np.mean(acc)
    

NAME = 'train_mhc_ba_el_in-silico_pci_db'
EPOCHS = 30
BATCH_SIZE = 4
MAX_PEPTIDES_TRAIN = 1_000

print(NAME)
for fold, (train_val_donors, _) in enumerate(splits):
    print(f'Fold {fold}')

    tue_train_data = get_hetero_data(train_val_donors)

    model = GNN(
        vocab_size=45,
        embedding_dim=128,
        dim_ff_enc_pep=128,
        n_heads_enc_pep=8,
        n_layers_enc_pep=6,
        dim_ff_enc_mhc=1024,
        n_heads_enc_mhc=8,
        n_layers_enc_mhc=6,
        n_heads_conv=8,
        n_layers_conv=2,
        dim_out_conv=32,
        dropout=0.1,
        act=F.leaky_relu,
        device1=DEVICE1,
        device2=DEVICE2,
    )

    state_dict = torch.load('weights/in_silico/pretrain_mhc_ba_el_in-silico.pt')
    model.load_state_dict(state_dict, strict=True)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # Training loop
    for epoch in range(1, EPOCHS+1):
        train_batched_data = [
            b.subgraph({
                'peptide': np.random.choice(
                    len(b['peptide'].x), min(int(len(b['peptide'].x)), MAX_PEPTIDES_TRAIN), replace=False
                )
            }) for b in tue_train_data
        ]
        train_loader = DataLoader(train_batched_data, batch_size=BATCH_SIZE, shuffle=True)
        model.train()
        batches = tqdm(train_loader)
        bl, train_acc = [], []
        for batch in batches:
            optimizer.zero_grad()
            out = model(batch)
            loss = F.binary_cross_entropy_with_logits(out, batch['mhc'].y.to(DEVICE2))
            train_acc.append(get_accuracy(batch, out))
            bl.append(loss.item())
            loss.backward()
            optimizer.step()
            descr = f'Train epoch {epoch} - loss: {np.mean(bl[-100:]):.4f}, acc: {np.mean(train_acc[-100:]):.4f}'
            batches.set_description(descr)
           
    torch.save(model.state_dict(), f'weights/{NAME}_{fold}.pt')

### Inference

In [ ]:
NAME = 'train_mhc_ba_el_in-silico_pci_db'

print(NAME)
preds, fold_donor_df = [], []
for fold, (_, test_donors) in enumerate(splits):
    print(f'Fold {fold}')
    model = GNN(
        vocab_size=45,
        embedding_dim=128,
        dim_ff_enc_pep=128,
        n_heads_enc_pep=8,
        n_layers_enc_pep=6,
        dim_ff_enc_mhc=1024,
        n_heads_enc_mhc=8,
        n_layers_enc_mhc=6,
        n_heads_conv=8,
        n_layers_conv=2,
        dim_out_conv=32,
        dropout=0.1,
        act=F.leaky_relu,
        device1=DEVICE1,
        device2=DEVICE2,
    )
    state_dict = torch.load(f'weights/{NAME}_{fold}.pt')
    model.load_state_dict(state_dict, strict=True)

    tue_test_data = get_hetero_data(test_donors)
    test_loader = DataLoader(tue_test_data, batch_size=1, shuffle=False)
    
    model.eval()
    with torch.no_grad():
        batches = tqdm(test_loader)
        for batch in batches:
            out = model(batch)  # TODO: change lat
            preds.append([out.cpu().detach().numpy()])
            fold_donor_df.append([batch.donor_code[0], batch.sample_code[0], fold])

fold_df = pd.concat([
    pd.DataFrame(fold_donor_df, columns=['donor_code', 'sample_code', 'fold']),
    pd.DataFrame(np.squeeze(np.concatenate(preds)), columns=donor_df.columns)
], axis=1)

pd.merge(fold_df, tue_db[['donor_code', 'sample_code', 'tissue']].drop_duplicates()).to_parquet(
    f'predictions/{NAME}.parquet', index=False
)

# Total

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# works only with batch size >= 2
def get_accuracy(batch, pred):
    acc = []
    ptr = batch['mhc']['ptr'][1:]
    for i in range(1, len(ptr)):
        p = pred[ptr[i-1]:ptr[i]].cpu().detach().numpy().squeeze()
        y = batch['mhc'].y[ptr[i-1]:ptr[i]].cpu().detach().numpy().squeeze()
        acc.append(len(set(np.argsort(y)[-6:]).intersection(set(np.argsort(p)[-6:]))) / 6)
    return np.mean(acc)


NAME = 'train_mhc_ba_el_in-silico_pci_db'
EPOCHS = 30
BATCH_SIZE = 4
MAX_PEPTIDES_TRAIN = 1_000


tue_train_data = get_hetero_data(list(donor_df.index))

model = GNN(
    vocab_size=45,
    embedding_dim=128,
    dim_ff_enc_pep=128,
    n_heads_enc_pep=8,
    n_layers_enc_pep=6,
    dim_ff_enc_mhc=1024,
    n_heads_enc_mhc=8,
    n_layers_enc_mhc=6,
    n_heads_conv=8,
    n_layers_conv=2,
    dim_out_conv=32,
    dropout=0.1,
    act=F.leaky_relu,
    device1=DEVICE1,
    device2=DEVICE2,
)

state_dict = torch.load('weights/in_silico/pretrain_mhc_ba_el_in-silico.pt')
model.load_state_dict(state_dict, strict=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(NAME)
# Training loop
for epoch in range(1, EPOCHS+1):
    train_batched_data = [
        b.subgraph({
            'peptide': np.random.choice(
                len(b['peptide'].x), min(int(len(b['peptide'].x)), MAX_PEPTIDES_TRAIN), replace=False
            )
        }) for b in tue_train_data
    ]
    train_loader = DataLoader(train_batched_data, batch_size=BATCH_SIZE, shuffle=True)
    model.train()
    batches = tqdm(train_loader)
    bl, train_acc = [], []
    for batch in batches:
        optimizer.zero_grad()
        out = model(batch)
        loss = F.binary_cross_entropy_with_logits(out, batch['mhc'].y.to(DEVICE2))
        train_acc.append(get_accuracy(batch, out))
        bl.append(loss.item())
        loss.backward()
        optimizer.step()
        descr = f'Train epoch {epoch} - loss: {np.mean(bl[-100:]):.4f}, acc: {np.mean(train_acc[-100:]):.4f}'
        batches.set_description(descr)
       
torch.save(model.state_dict(), f'weights/{NAME}.pt')